# ANALYSIS

### Ingesting

In [1]:
from pyspark.sql import SparkSession
from pathlib import Path

file_path = Path.cwd()
data_path = file_path.parent / "data" / "e-shop_clothing_2008.csv"

spark = SparkSession.builder.appName("clickstream-exploration").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv(
    str(data_path),
    header=True,
    inferSchema=True,
    sep=";"
)
df.printSchema()

26/04/04 10:31:16 WARN Utils: Your hostname, Sajjads-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.108 instead (on interface en0)
26/04/04 10:31:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/04 10:31:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session ID: integer (nullable = true)
 |-- page 1 (main category): integer (nullable = true)
 |-- page 2 (clothing model): string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price 2: integer (nullable = true)
 |-- page: integer (nullable = true)



## Cleaning

### Removing odd characters

In [2]:
import re

for col in df.columns:
    new_name = re.sub(r'[ ()/]+', '_', col).strip('_').lower()
    df = df.withColumnRenamed(col, new_name)

df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- page_1_main_category: integer (nullable = true)
 |-- page_2_clothing_model: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model_photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_2: integer (nullable = true)
 |-- page: integer (nullable = true)



### Merging day, month,year into DATE

In [6]:
from pyspark.sql import functions as F

df = df.withColumn(
    "date",
    F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.col("day")), "yyyy-M-d")
).drop("year", "month", "day")

In [7]:
df.head(1)

[Row(order=1, country=29, session_id=1, page_1_main_category=1, page_2_clothing_model='A13', colour=1, location=5, model_photography=1, price=28, price_2=2, page=1, date=datetime.date(2008, 4, 1))]

### Handling catagories

In [8]:
from pyspark.sql.types import StringType

In [9]:
df = df.withColumn("page_1_main_category", F.col("page_1_main_category").cast(StringType()))
df = df.withColumn("colour", F.col("colour").cast(StringType()))
df = df.withColumn("location", F.col("location").cast(StringType()))
df = df.withColumn("model_photography", F.col("model_photography").cast(StringType()))

In [10]:
df = df.withColumn("price_2", F.col("price_2") == 1)

### Checking for nulls

In [11]:
from pyspark.sql.functions import col, sum as spark_sum

df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+
|order|country|session_id|page_1_main_category|page_2_clothing_model|colour|location|model_photography|price|price_2|page|date|
+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+
|    0|      0|         0|                   0|                    0|     0|       0|                0|    0|      0|   0|   0|
+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+



### Checking for repeated *session_ids* or *orders*

In [12]:
df.groupBy("session_id", "order").count().filter(F.col("count") > 1).show()

+----------+-----+-----+
|session_id|order|count|
+----------+-----+-----+
+----------+-----+-----+



### Saving the cleaned df

In [13]:
output_path = Path.cwd().parent / "output" / "cleaned.parquet"
df.write.parquet(str(output_path), mode="overwrite")

# SESSIONizing

## number of sessions

In [15]:
df.select ("session_id").distinct().count ()

24026

## number of clicks per session

In [16]:
df.groupBy("session_id").agg(
    F.count("*").alias("n_clicks"),
    F.mean ("price").alias ("avg_price"),
    F.countDistinct("page_1_main_category").alias("n_categories"),
    F.countDistinct("colour").alias("n_colours"),
).show ()

+----------+--------+------------------+------------+---------+
|session_id|n_clicks|         avg_price|n_categories|n_colours|
+----------+--------+------------------+------------+---------+
|       496|       2|              45.0|           1|        2|
|      5803|       4|              35.5|           1|        4|
|      9900|       7|55.142857142857146|           1|        6|
|      3794|       9|43.888888888888886|           3|        6|
|      3997|      11| 39.36363636363637|           2|        7|
|      8592|       3|              41.0|           1|        2|
|      5518|      18| 37.72222222222222|           3|        9|
|      6336|       9|37.666666666666664|           3|        5|
|      7754|       7|46.714285714285715|           2|        4|
|      6466|       1|              33.0|           1|        1|
|     10362|      10|              38.7|           1|        6|
|      6658|      11| 35.54545454545455|           3|        6|
|     11858|       3|41.333333333333336|

In [17]:
from pyspark.sql import Window

category_counts = df.groupBy("session_id", "page_1_main_category").count()

window = Window.partitionBy("session_id").orderBy(F.col("count").desc())

dominant_category = category_counts.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") == 1) \
    .select("session_id", F.col("page_1_main_category").alias("dominant_category"))

dominant_category.show(5)

+----------+-----------------+
|session_id|dominant_category|
+----------+-----------------+
|         1|                2|
|         2|                2|
|         3|                3|
|         4|                3|
|         5|                3|
+----------+-----------------+
only showing top 5 rows



In [47]:
category_counts.filter (F.col("session_id")== 1).show ()

+----------+--------------------+-----+
|session_id|page_1_main_category|count|
+----------+--------------------+-----+
|         1|                   3|    2|
|         1|                   1|    2|
|         1|                   4|    2|
|         1|                   2|    3|
+----------+--------------------+-----+



In [18]:
cleaned_path = Path.cwd().parent / "output" / "cleaned.parquet"
df = spark.read.parquet(str(cleaned_path))

## aggregate per session

In [ ]:
sessions = df.groupBy("session_id").agg(
    F.count("*").alias("n_clicks"),
    F.avg("price").alias("avg_price"),
    F.countDistinct("page_1_main_category").alias("n_categories"),
    F.countDistinct("colour").alias("n_colours"),
    F.max("price_2").alias("bought"),   # price_2 is True if discounted/bought
    F.first("country").alias("country"),
    F.first("date").alias("date")
)

sessions.show(5)